<a href="https://colab.research.google.com/github/vee234o/c10-team-imo/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load dataset
clean data
vectorize
train model to detect whether there is stress or not
test
evaluate


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()
gc.collect()

print("CUDA memory allocator configured and cache cleared.")

In [ ]:
import pandas as pd
import numpy as np
import re
import html
import torch
import joblib
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
pd.set_option('display.max_colwidth', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_df= pd.read_csv('/content/drive/My Drive/Data/dreaddit-train.csv')
test_df= pd.read_csv('/content/drive/My Drive/Data/dreaddit-test.csv')

In [ ]:
train_df.info()

In [ ]:
train_df = train_df[['text','label']]
test_df = test_df[['text','label']]

In [ ]:
def clean_social_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = text.replace(r"\'", "'").replace(r'"', '"').replace(r"\\", "")

    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\b[ru]/", "", text)
    text = re.sub(r"(?i)\b(trigger\s*(warning|ahead)[^:]*:?|tw\s*[:\-])", "", text)

    text = re.sub(r"^\s*>\s?", "", text, flags=re.MULTILINE)
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^	op*\s*[\*\-\+]\s+", "", text, flags=re.MULTILINE)

    text = re.sub(r"(\*\*|__)(.*?)\1", r"\2", text)
    text = re.sub(r"(\*|_)(.*?)\1", r"\2", text)

    text = re.sub(r"\.{2,}", "...", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:

train_df['text'] = train_df['text'].apply(clean_social_text)
test_df['text'] = test_df['text'].apply(clean_social_text)


In [ ]:
train_df.isna().sum()

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
def extract_layer_23(texts, batch_size=4):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        ).to("cpu")

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            layer_23_hidden = outputs.hidden_states[23]

            attention_mask = inputs["attention_mask"].unsqueeze(-1)
            masked_hidden = layer_23_hidden * attention_mask
            summed = torch.sum(masked_hidden, dim=1)
            counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_pooled = summed / counts

            all_embeddings.append(mean_pooled.cpu().float())

    return torch.cat(all_embeddings, dim=0).numpy()

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModel

print("Loading Gemma-2-2B model on CPU...")
model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    model_id,
    output_hidden_states=True,
    dtype=torch.float32,
    device_map="cpu"
)
model.eval()
print("Model loaded successfully")

In [ ]:

train_texts = train_df['text'].tolist()
test_texts = test_df['text'].tolist()


train_labels = train_df['label'].values
test_labels = test_df['label'].values

print("\nExtracting training embeddings...")
X_train = extract_layer_23(train_texts)

print("Extracting testing embeddings...")
X_test = extract_layer_23(test_texts)


print(f"\nX_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


Extracting training embeddings...


In [ ]:
from sklearn.svm import SVC
import joblib

print("Training Support Vector Classifier (SVC)...")
svc_probe = SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42)
svc_probe.fit(X_train, train_labels)
joblib.dump(svc_probe, 'stress_svc_probe.joblib')
print("SVC model trained and saved successfully!")

In [ ]:
import joblib
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

loaded_model = joblib.load('stress_svc_probe.joblib')
loaded_scaler = joblib.load('feature_scaler.joblib')

test_texts = test_df['text'].tolist()
test_labels = test_df['label'].values

X_test = extract_layer_23(test_texts, batch_size=4)
X_test_scaled = loaded_scaler.transform(X_test)
test_preds = loaded_model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(test_labels, test_preds):.4f}\n")
print("Classification Report:")
print(classification_report(test_labels, test_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))